In [1]:
from openai import OpenAI
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

client = OpenAI()


In [3]:
response = client.responses.create(
    model="gpt-4.1-mini",
    tools=[{"type": "web_search"}],
    input="2026년 전 세계에서 있었던 좋은 뉴스 하나 알려줘."
)

print(response.output_text)
print(response.output[0])

2026년에는 전 세계적으로 여러 긍정적인 소식이 있었습니다. 그 중 하나는 **세계적인 빈곤율 감소**입니다. 국제연합(UN)의 최근 보고서에 따르면, 2026년까지 전 세계 극단적 빈곤율이 10% 이하로 감소하여, 1990년대 초반 이후 가장 큰 개선을 보였습니다. 이는 국제 사회의 지속적인 노력과 개발 프로그램의 효과적인 실행 덕분입니다.

또한, **재생 가능 에너지의 급속한 성장**도 주목할 만한 성과입니다. 2026년에는 전 세계 에너지 소비에서 재생 가능 에너지의 비중이 40%를 넘어섰으며, 이는 기후 변화 대응과 지속 가능한 발전을 위한 중요한 진전을 의미합니다.

이러한 긍정적인 변화들은 국제 사회의 협력과 지속적인 노력의 결과로, 앞으로도 더 많은 발전이 기대됩니다. 
ResponseFunctionWebSearch(id='ws_054f646dec2efa41006a87b566615c87d0bf3151e0f9d3e420', action=ActionSearch(type='search', queries=['positive global news 2026'], query='positive global news 2026', sources=None), status='completed', type='web_search_call')


In [ ]:
client.responses.create(
    model="gpt-4.1",
    input="우리 제품 환불 규정 요약해줘.",
    tools=[{
        "type": "file_search",
        "vector_store_ids": ["<vector_store_id>"],
    }],
)

In [4]:
import base64

response = client.responses.create(
    model="gpt-4.1-mini",
    input="다 먹은 제철 감자칩 그림 그려줘.",
    tools=[{"type": "image_generation"}],
)

# Save the image to a file
image_data = [
    output.result
    for output in response.output
    if output.type == "image_generation_call"
]

if image_data:
    image_base64 = image_data[0]
    with open("pic.png", "wb") as f:
        f.write(base64.b64decode(image_base64))

In [6]:
instructions = """
You are a personal math tutor. When asked a math question,
write and run code using the python tool to answer the question.
"""

resp = client.responses.create(
    model="gpt-4.1-mini",
    tools=[
        {
            "type": "code_interpreter",
            "container": {"type": "auto", "memory_limit": "4g"}
        }
    ],
    instructions=instructions,
    input="I need to solve the equation 3x + 11 = 14. Can you help me?",
)

print(resp.output)

for output in resp.output:
  if output.type =="message":
    print(output.content[0].text)

[ResponseOutputMessage(id='msg_0a133e3ac1fbcfbb006a87b8601d9887d0b43e3982fb562a80', content=[ResponseOutputText(annotations=[], text="Sure! To solve the equation \\(3x + 11 = 14\\), we will isolate \\(x\\) by performing algebraic operations step by step.\n\nLet's solve it.", type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None), ResponseCodeInterpreterToolCall(id='ci_0a133e3ac1fbcfbb006a87b860f4b887d09c9e9847e40b69d2', code="from sympy import symbols, Eq, solve\r\n\r\n# Define the variable\r\nx = symbols('x')\r\n\r\n# Define the equation\r\nequation = Eq(3*x + 11, 14)\r\n\r\n# Solve the equation\r\nsolution = solve(equation, x)\r\nsolution", container_id='cntr_6a87b85f801c81989163a26813f6fa6b01dcb2f3476abbd4', outputs=None, status='completed', type='code_interpreter_call')]
Sure! To solve the equation \(3x + 11 = 14\), we will isolate \(x\) by performing algebraic operations step by step.

Let's solve it.


In [8]:
tools = [
    {
        "type": "function",
        "name": "get_order_status",
        "description": "주문 ID를 받아 현재 배송 상태를 조회한다.",
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string"},
            },
            "required": ["order_id"],
        },
    },
    {
        "type": "function",
        "name": "create_support_ticket",
        "description": "고객 문의/불만을 티켓으로 생성한다.",
        "parameters": {
            "type": "object",
            "properties": {
                "title": {"type": "string"},
                "description": {"type": "string"},
                "priority": {
                    "type": "string",
                    "enum": ["low", "normal", "high"],
                },
            },
            "required": ["title", "description"],
        },
    },
]


In [7]:
def get_order_status(order_id: str):
    return {"status": "shipped"}

def create_support_ticket(title: str, description: str, priority: str):
    return {"ticket_id": "12345", "priority" : priority}

In [9]:
import json

def handle_tool_call(item):
    args = json.loads(item.arguments)
    print(args)
    if item.name == "get_order_status":
        return get_order_status(**args)
    elif item.name == "create_support_ticket":
        return create_support_ticket(**args)
    else:
        return {"error": f"Unknown tool {item.name}"}



def run_support_agent(user_message: str):
    # 1차 호출: 어떤 툴을 쓸지 모델에게 맡김
    messages = [{"role": "user", "content": user_message}]
    resp = client.responses.create(
        model="gpt-4.1-mini",
        tools=tools,
        instructions = "사용자 요청에 대해 적절한 도구를 선택하되, 불만/요청 등의 문구가 있으면 create_support_ticket 도구를 사용해",
        input=messages,
    )

    messages += list(resp.output)  # function_call 들을 히스토리에 추가

    # 여러 function_call에 대해 반복
    for item in resp.output:
        if item.type == "function_call":
            print(item)
            result = handle_tool_call(item)
            messages.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": json.dumps(result, ensure_ascii=False),
            })

    # 최종 답변 생성
    resp2 = client.responses.create(
        model="gpt-4.1-mini",
        tools=tools,
        input=messages,
        instructions=(
            "너는 고객센터 상담사야. function_call_output으로 전달된 "
            "주문/티켓 정보를 활용해서, 상황을 정리해주고 다음 액션을 제안해줘.티켓이 생성된 경우 티켓 번호를 알려줘야 해."
        ),
    )
    print(resp2.output_text)




In [10]:
run_support_agent("주문 2024-000123 상태 좀 알려줘.")

ResponseFunctionToolCall(arguments='{"order_id":"2024-000123"}', call_id='call_RPZmusvgfwPyJNspw4Nm2MMN', name='get_order_status', type='function_call', id='fc_005d4bbfdb7ee985006a87bc663c3087d09a37e3b088343628', caller=None, namespace=None, status='completed')
{'order_id': '2024-000123'}
주문번호 2024-000123 의 현재 배송 상태는 '발송(shipped)'되었습니다. 추가로 궁금한 점이 있거나 도움이 필요하시면 알려주세요.


In [12]:
run_support_agent("앱이 자꾸 튕겨서 불만 접수 티켓 만들어줘.")
#run_support_agent("앱 배경 색이 좀 거슬리는데 수정 요청하고 싶어.")

ResponseFunctionToolCall(arguments='{"title":"앱 튕김 현상 신고","description":"사용자가 앱이 자꾸 튕긴다는 불만을 제기함.","priority":"high"}', call_id='call_BF1zbXaDFnci9jsYI3ObB1j2', name='create_support_ticket', type='function_call', id='fc_0189d54e35313c2a006a87bcccf15c87d0b6e1bb0c901aa194', caller=None, namespace=None, status='completed')
{'title': '앱 튕김 현상 신고', 'description': '사용자가 앱이 자꾸 튕긴다는 불만을 제기함.', 'priority': 'high'}
앱이 자꾸 튕기는 문제로 불만 접수 티켓을 생성했습니다. 티켓 번호는 12345입니다. 담당 팀에서 빠르게 문제를 조사하고 해결할 수 있도록 하겠습니다. 추가로 궁금한 점이나 불편한 점 있으면 언제든 말씀해 주세요.


In [13]:
run_support_agent("주문 2024-000123 너무 늦게 오고, 환불 불만 티켓도 같이 남겨줘.")

ResponseFunctionToolCall(arguments='{"order_id":"2024-000123"}', call_id='call_0B6GNI2k5t5R4OYd7FTKeaxL', name='get_order_status', type='function_call', id='fc_071746f07f05d3e5006a87bdca1a4c87d0a982f051ddda83ce', caller=None, namespace=None, status='completed')
{'order_id': '2024-000123'}
ResponseFunctionToolCall(arguments='{"title":"주문 2024-000123 배송 지연 및 환불 불만","description":"주문 번호 2024-000123이 너무 늦게 배송되고 있어 불만입니다. 환불 요청도 함께 처리해 주세요.","priority":"high"}', call_id='call_X8C82CekwryEmEWHtCBnQHtY', name='create_support_ticket', type='function_call', id='fc_071746f07f05d3e5006a87bdca1a5c87d0b498cf7512da4cd9', caller=None, namespace=None, status='completed')
{'title': '주문 2024-000123 배송 지연 및 환불 불만', 'description': '주문 번호 2024-000123이 너무 늦게 배송되고 있어 불만입니다. 환불 요청도 함께 처리해 주세요.', 'priority': 'high'}
주문 번호 2024-000123은 현재 배송 중 상태입니다. 하지만 배송이 너무 늦어진 점에 대해 불만이 접수되었고, 환불 요청도 함께 티켓 번호 12345로 생성되었습니다. 고객님의 불편을 신속히 해결하기 위해 담당 부서에서 우선적으로 처리할 예정입니다. 추가 문의 사항이 있으시면 언제든 알려주세요.


In [14]:
mixed_tools = [
    {"type": "web_search"},
    {
        "type": "function",
        "name": "save_insight",
        "description": "중요한 리서치 인사이트를 내부 시스템에 저장한다.",
        "parameters": {
            "type": "object",
            "properties": {
                "topic": {"type": "string"},
                "sentiment": {"type": "string", "enum": ["positive", "negative", "neutral"]},
                "summary": {"type": "string"},
            },
            "required": ["topic", "sentiment", "summary"],
        },
    },
]

In [15]:
def fake_save_insight(topic, sentiment, summary):
    print(f"[저장됨] topic={topic}, sentiment={sentiment}")
    print(f"summary={summary[:80]}...")
    return {"status": "ok"}

def handle_mixed_tool_call(item):
    import json
    args = json.loads(item.arguments)
    if item.name == "save_insight":
        return fake_save_insight(**args)
    else:
        return {"error": f"unknown tool {item.name}"}

def run_research_agent(question: str):
    messages = [{"role": "user", "content": question}]
    resp = client.responses.create(
        model="gpt-5",
        tools=mixed_tools,
        input=messages,
    )
    messages += list(resp.output)

    # function_call만 우리가 처리, web_search는 OpenAI 쪽에서 이미 처리 완료
    for item in resp.output:
        if item.type == "function_call":
            print(item)
            result = handle_mixed_tool_call(item)

            messages.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": json.dumps(result, ensure_ascii=False),
            })

    resp2 = client.responses.create(
        model="gpt-5",
        tools=mixed_tools,
        input=messages,
        instructions=(
            "웹 검색과 save_insight 함수를 적절히 사용해서 "
            "사용자 질문에 답하고, 중요한 인사이트는 저장해라."
        ),
    )
    print(resp2.output_text)

run_research_agent(" Apple 관련 긍정적인 뉴스 하나를 웹에서 찾아서 요약하고, 인사이트도 저장해줘.")

ResponseFunctionToolCall(arguments='{"topic":"Apple: Houston Advanced Manufacturing Center","sentiment":"positive","summary":"애플이 2026년 8월 휴스턴에 ‘첨단 제조 센터(AMC)’를 열어 중소기업과 학생에게 무료 스마트 제조 교육을 제공하고, 같은 부지에서 AI 서버를 생산하며 올해 안에 Mac mini 생산까지 예고했습니다. 이는 미국 내 제조·공급망·인재 기반을 강화해 정책 리스크를 줄이고, 하드웨어와 AI 인프라 역량을 확대하려는 전략적 투자로 보입니다."}', call_id='call_wvXZcUWuDmtlZpRLFH1Y0kW3', name='save_insight', type='function_call', id='fc_0768336a03b2f4b2006a87be8c7f1487d0a149503cf4acd8b8', caller=None, namespace=None, status='completed')
[저장됨] topic=Apple: Houston Advanced Manufacturing Center, sentiment=positive
summary=애플이 2026년 8월 휴스턴에 ‘첨단 제조 센터(AMC)’를 열어 중소기업과 학생에게 무료 스마트 제조 교육을 제공하고, 같은 부지에서 AI ...
다음은 최근 애플 관련 긍정적 뉴스 1건 요약입니다.

- 제목/날짜: “Apple opens Advanced Manufacturing Center in Houston” (2026년 8월 13일)
- 핵심 요약: 애플이 미국 텍사스 휴스턴에 ‘첨단 제조 센터(AMC)’를 개소했습니다. 이 센터는 중소기업과 학생에게 무료로 스마트 제조 교육을 제공하고, 동일 부지에서 이미 고급 AI 서버를 출하 중이며 올해 안에 Mac mini 생산도 시작할 계획입니다. 애플의 미국 내 두 번째 제조 학습 거점으로, 2만 제곱피트 규모의 시설과 최신 장비·실습 환경을 갖췄습니